# 这是一个使用langchain的agent 例子，使用Azure OpenAI

In [4]:
! pip install langchain
! pip install langchain_openai

In [18]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain.agents import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents.format_scratchpad.openai_tools import (
    format_to_openai_tool_messages,
)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser
from langchain.agents import AgentExecutor
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
import asyncio
from typing import Any

# Load environment variables from .env file
load_dotenv("/etc/.env")

# Set the OpenAI API key and endpoint
deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT")
api_base=os.getenv("AZURE_OPENAI_API_BASE")
endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
api_key=os.getenv("AZURE_OPENAI_API_KEY")
api_version=os.getenv("AZURE_OPENAI_API_VERSION","2025-03-01-preview")

# Initialize the AzureChatOpenAI model
llm =  AzureChatOpenAI(
    openai_api_version=api_version,
    openai_api_base=api_base,
    openai_api_key=api_key,
    azure_endpoint=endpoint,
    deployment_name=deployment,
)

tool_usage_log = []

def log_tool_usage(tool_name: str, input_data: Any):
    """Logs the tool used and its input."""
    tool_usage_log.append({"tool": tool_name, "input": input_data})


@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    log_tool_usage("get_word_length", word)
    return len(word)

@tool
def calculator(expression :str) -> str:
    """Evaluates mathematical expression"""
    log_tool_usage("calculator", expression)
    try:
        maths_result = eval(expression)
        return str(maths_result)
    except Exception as e:
        return f"Error: {str(e)}"

tools = [get_word_length, calculator]
llm_with_tools = llm.bind_tools(tools)

MEMORY_KEY = "chat_history"

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are very powerful assistant, but bad at calculating lengths of words and mathematical expressions",
        ),
        MessagesPlaceholder(variable_name=MEMORY_KEY),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

async def main():
    chat_history = []
    while True:
        print("Enter question or type exit to quit")
        input1 = input("User: ")

        if input1.lower() == "exit":
            print("Exiting the chat.")
            break

        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(None, lambda: agent_executor.invoke({"input":input1 , "chat_history": chat_history}))

        chat_history.extend(
        [
            HumanMessage(content=input1),
            AIMessage(content=result["output"]),
        ]
    )
        print("\nTools Used:")
        for usage in tool_usage_log:
            print(f"Tool: {usage['tool']}, Input: {usage['input']}")
            
        print("\n\n Message:\n", result["output"])

# if __name__ == "__main__":
#     asyncio.run(main())


await main()


Enter question or type exit to quit

Tools Used:
Tool: calculator, Input: 5+7


 Message:
 5+7=12
Enter question or type exit to quit

Tools Used:
Tool: calculator, Input: 5+7


 Message:
 如果你是想让我们计算数学表达式“5+7”，那么答案是 12。如果需要进一步的帮助，请告诉我！
Enter question or type exit to quit
Exiting the chat.


In [24]:
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

template = "You are a helpful assistant that translates {input_language} to {output_language}."
system_message_prompt = SystemMessagePromptTemplate.from_template(template)
human_template = "{text}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

message = chat_prompt.format_messages(input_language="English", output_language="Chinese", text="I love programming.")

response = llm.invoke(message)
print(response.content)

print(response)

我热爱编程。
content='我热爱编程。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 26, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BQb8yoZFhde9ZDLhAMUFXB9g5ukld', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'filtered': False, 'detected'